### Import libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity


### Load Datasets

In [2]:
movies_filtered = pd.read_csv("movies_filtered.csv")

### Create the content matrix

In [3]:
all_genres = set()
for genres_str in movies_filtered['genres']:
    all_genres.update(genres_str.split('|'))
all_genres = sorted(all_genres)

content_matrix = pd.DataFrame(0, index=movies_filtered['movieId'], columns=all_genres)
for idx, row in movies_filtered.iterrows():
    content_matrix.loc[row['movieId'], row['genres'].split('|')] = 1

In [4]:
content_matrix

,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
movieId,,,,,,,,,,,,,,,,,,,
1,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292731,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
292737,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
292753,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [6]:
content_matrix.shape

(80505, 19)

In [7]:
def clustering(content_matrix):

    best_k = 2

    best_silhouette = -1

    silhouette_scores = []

    for k in range (2,11):
            
            kmeans = KMeans(n_clusters=k, random_state=42)
            cluster_labels = kmeans.fit_predict(content_matrix)

            silhouette_avg = silhouette_score(content_matrix, cluster_labels)
            silhouette_scores.append((k, silhouette_avg))

            if silhouette_avg > best_silhouette:
                best_k = k
                best_silhouette = silhouette_avg
    return best_k, silhouette_scores

best_k, silhouette_scores = clustering(content_matrix)

print(f"Optimal number of clusters: {best_k}")
print("Silhouette scores for each k:", silhouette_scores)

Optimal number of clusters: 10
Silhouette scores for each k: [(2, 0.20488036959946831), (3, 0.23515629743014538), (4, 0.2559542452273855), (5, 0.3164505509966757), (6, 0.33158462698327773), (7, 0.34030654318394), (8, 0.37088032649295577), (9, 0.39705914467879766), (10, 0.413736036419257)]


In [8]:
kmeans = KMeans(n_clusters=best_k, random_state=42)

content_matrix["cluster"] = kmeans.fit_predict(content_matrix)

In [9]:
content_matrix.to_csv("content_matrix.csv") 

In [23]:
def content_based_recommendation(movie_title, top_n=5):
    # Trouver l'index du film dans movies_filtered
    idx = movies_filtered[movies_filtered['title'] == movie_title].index[0]
    movie_id = movies_filtered.loc[idx, 'movieId']

    # Trouver le cluster du film dans content_matrix (indexé par movieId)
    cluster = content_matrix.loc[movie_id, 'cluster']

    # Sélectionner les movieId du même cluster (hors le film de référence)
    same_cluster_ids = content_matrix[content_matrix['cluster'] == cluster].index
    same_cluster_ids = same_cluster_ids[same_cluster_ids != movie_id]

    # Calculer la similarité cosinus
    ref_vector = content_matrix.loc[movie_id].drop('cluster').values.reshape(1, -1)
    cluster_matrix = content_matrix.loc[same_cluster_ids].drop('cluster', axis=1).values

    similarities = cosine_similarity(ref_vector, cluster_matrix)[0]
    top_indices = similarities.argsort()[::-1][:top_n]
    recommended_ids = same_cluster_ids[top_indices]

    # Afficher les films recommandés
    recommended_movies = movies_filtered[movies_filtered['movieId'].isin(recommended_ids)]
    print("Films recommandés :")
    print(recommended_movies[['title', 'genres']])

# Exemple d'utilisation
content_based_recommendation("Man of Steel (2013)", top_n=5)

Films recommandés :
                                                   title  \
4171                                        Krull (1983)   
17067                               Avengers, The (2012)   
20489                                Ender's Game (2013)   
25026  Star Wars: Episode VII - The Force Awakens (2015)   
70811                   Justice League of America (1997)   

                                     genres  
4171        Action|Adventure|Fantasy|Sci-Fi  
17067          Action|Adventure|Sci-Fi|IMAX  
20489          Action|Adventure|Sci-Fi|IMAX  
25026  Action|Adventure|Fantasy|Sci-Fi|IMAX  
70811       Action|Adventure|Fantasy|Sci-Fi  
